In [1]:
# api = lang req - llm = answer 
# api - llm - answer # paid - prot open ai, groq 
#llm 
from dotenv import load_dotenv
import os            # os.getenv("GOOGLE_API_KEY")
from langchain_google_genai import ChatGoogleGenerativeAI      # the integration package for using Google's generative AI models through LangChain.ChatGoogleGenerativeAI — the class used to configure a chat model, such as Gemini.
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate
import requests # api get 
from langchain_core.tools import tool # Decorator that lets us convert Python functions into tools that agent can use.
from langchain.agents import create_agent #create_agent → creates an AI agent that uses an LLM to perform tasks.

#langchain_core  --> (messages -> HumanMessage,,,prompts -> PromptTemplate,,,tools -> tool)

In [2]:
load_dotenv()
a = os.getenv("GEMINI_API_KEY")      # this will print you api key but it should be private...
print(a is not None)

True


In [3]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    project="project-d24e3f00-1d87-4332-818",
    location="global",
    vertexai=True,
    temperature=0.7
) #this code creates your LLM (Gemini AI model) so your agent can use it.

In [4]:
#api call 
API="https://681444af225ff1af16285b31.mockapi.io/electronicapi"    

#API = a variable holding the URL.
# mockapi.io — a service for creating fake API endpoints for testing.
#Your Python code can use this URL with requests to fetch data from that API, for example, using requests.get(API).

In [5]:
APIS="https://681444af225ff1af16285b31.mockapi.io/n8napi"

In [6]:
# this function creates a tool that allows your AI agent to fetch electronics data from your API.

@tool        # Decorator that lets us convert Python functions into tools that agent can use.
def electronics():
    """ u r electronics api experts"""            # is a docstring , It describes what the electronics() function does and helps the agent understand the tool's purpose.
    response=requests.get(API)                    # sends an HTTP GET request to your API URL.

    return response.json()                        #converts the API's JSON response into Python data and returns it.

#--------------------------------------------------------------------
''' THIS IS FOR TO GET SPECIFIC ITEM FROM API...
@tool
def electronics(product_name: str):
    """Fetch details of a specific electronics product by name."""
    response = requests.get(API)
    data = response.json()

    return [
        item for item in data
        if product_name.lower() in item.get("name", "").lower()
    ]


result = electronics.invoke({"product_name": "iPhone"})
result

'''

' THIS IS FOR TO GET SPECIFIC ITEM FROM API...\n@tool\ndef electronics(product_name: str):\n    """Fetch details of a specific electronics product by name."""\n    response = requests.get(API)\n    data = response.json()\n\n    return [\n        item for item in data\n        if product_name.lower() in item.get("name", "").lower()\n    ]\n\n\nresult = electronics.invoke({"product_name": "iPhone"})\nresult\n\n'

In [7]:
@tool
def furniture():
    """ u r furniture api experts"""
    response=requests.get(APIS)

    return response.json()

In [8]:
agent= create_agent(                         #creates the agent.
    model=llm,                               # this model,tools,system_prompt this all are parameters of create_agent.
    tools=[electronics,furniture],
    system_prompt="""You are a helpful  assistant.

Use the electronics tool whenever the user asks
about available electronics products.

see the furniture tool whenever the user asks
about available furniture products.

Do not make up product information.
Use the tool data to answer.
"""
)  

In [9]:
question = input('ask replated product')


result=agent.invoke({
    "messages":[                     # why use message: --> our data isin json==key and value format...so to access it like that direct answer we write this.
        ("user",question)            # with agent.invoke(), you pass the question inside the expected messages format.
    ]})

print(result["messages"][-1].content[0]["text"])

The cheapest product in the electronics category is the **Samsung Galaxy Tablet** (listed under the name "Galaxy", ID: 53) with a price of **10**. 

Here are the top three cheapest products from the database:

1. **Galaxy** (Samsung Tablet) — **10**
2. **asdasd** (Canon Camera) — **400** *(likely a test entry)*
3. **JBL Tune 760NC** (Headphones) — **5,000**
